# Statistics for all Ojibwe treebanks

In [1]:
from collections import Counter
import pandas as pd
from grammar_modules.disambiguation import REPO_ROOT
from corpus_view import analyze_corpus, get_token_statistics

In [2]:
TREEBANK_ROOT = REPO_ROOT / "data" / "treebanks" / "AC"
treebank_files = sorted(TREEBANK_ROOT.glob("*.conllu"))

In [3]:
corpora = {}
token_statistics = {}

for filename in treebank_files:
    name = filename.stem[11:]
    print(f"Analyzing {name}...")
    corpora[name] = analyze_corpus(corpus_path=filename)
    token_statistics[name] = get_token_statistics(filename)

print("Finished.")

Analyzing Aanjikiing...
Analyzing Akawe_Niwii-tibaajim...
Analyzing Anooj_inaajimod...
Analyzing Awesiinyensag...
Analyzing Chi-mewinzha...
Analyzing Gii-nitaa-aadisooke...
Analyzing Living-our-language...
Analyzing Mino-doodaading...
Analyzing Naadamaading...
Analyzing Plums-or-nuts...
Analyzing Portage_Lake...
Analyzing The-dogs-children...
Analyzing Wiijikiiwending...
Analyzing example_sentences_opd...
Analyzing Dibaajimowinan...
Analyzing _Oshkaabewis...
Finished.


In [14]:
BASIC_UPOS_COLUMNS = (
    "VERB",
    "NOUN",
    "PROPN",
    "PRON",
    "ADV",
    "NUM",
    "DET",
)

basic_rows = []

for filename in treebank_files:
    name = filename.stem[11:]
    corpus = corpora[name]
    token_stats = token_statistics[name]
    upos_counts = token_stats["upos_counts"]
    sentence_count = len(corpus["sentences"])
    word_count = token_stats["word_count"]

    row = {
        "corpus": name,
        "sentences": sentence_count,
        "tokens": token_stats["token_count"],
        "words": word_count,
        "punctuation": token_stats["punctuation_count"],
        "unique_wordforms": token_stats["unique_word_forms"],
        "unique_lemmas": token_stats["unique_lemmas"],
    }
    row.update({upos.lower(): upos_counts[upos] for upos in BASIC_UPOS_COLUMNS})
    basic_rows.append(row)

basic_stats_df = pd.DataFrame(basic_rows).set_index("corpus")
count_columns = [
    "sentences",
    "tokens",
    "words",
    "punctuation",
    "unique_wordforms",
    "unique_lemmas",
    *(upos.lower() for upos in BASIC_UPOS_COLUMNS),
]

basic_stats_df[count_columns] = basic_stats_df[count_columns].astype("Int64")

total_row = {
    column: int(basic_stats_df[column].sum())
    for column in count_columns
}
total_row["unique_wordforms"] = pd.NA
total_row["unique_lemmas"] = pd.NA

basic_stats_df.loc["TOTAL"] = total_row
basic_stats_df

/var/folders/34/rqchv6x94fzd3gv3kf97vxf80000gq/T/ipykernel_98808/860193052.py:53: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  basic_stats_df.loc["TOTAL"] = total_row


,sentences,tokens,words,punctuation,unique_wordforms,unique_lemmas,verb,noun,propn,pron,adv,num,det
corpus,,,,,,,,,,,,,
Aanjikiing,304,2763,2344,419,731,448,726,251,2,165,879,7,205
Akawe_Niwii-tibaajim,1407,10991,8609,2382,3142,1353,2923,1245,8,507,2724,90,344
Anooj_inaajimod,337,2588,2043,545,919,578,694,294,3,93,690,21,94
Awesiinyensag,229,1560,1055,505,559,379,440,210,0,78,208,4,21
Chi-mewinzha,483,3186,2526,660,910,536,758,348,4,304,800,19,160
Gii-nitaa-aadisooke,461,2240,1592,648,722,515,567,251,0,75,467,9,105
Living-our-language,1433,10549,8141,2408,2424,1039,2371,881,32,826,2710,62,375
Mino-doodaading,199,1254,867,387,519,367,351,192,0,41,186,11,8
Naadamaading,289,1819,1228,591,623,417,470,284,3,56,259,12,28


In [5]:
all_predicates = [
    p
    for corpus in corpora.values()
    for p in corpus["predicates"]
]

In [ ]:
from collections import Counter

AN_THIRD_PERSON = {"3SgProx", "3PlProx", "3SgObv", "3PlObv"}

PERSON_HIERARCHY = {
    "2": 1, "1": 2, "E": 2, "I": 2,
    "X": 3, "3": 4, "0": 5
}

VTA_GROUPS = ("Non-local", "Local", "Mixed")
INFLECTIONAL_ORDERS = ("Ind", "Cnj", "Imp", "Pcp")

ARGUMENT_PATTERNS = ("Neither", "Subject", "Object", "Both")

WORD_ORDERS = {
    "Subject": ("SV", "VS"),
    "Object": ("VO", "OV"),
    "Both": ("SVO", "SOV", "VSO", "VOS", "OSV", "OVS"),
}


def argument_pattern(p):
    relations = {a["relation"] for a in p["arguments"]}
    subj, obj = "nsubj" in relations, "obj" in relations

    if subj and obj:
        return "Both"
    if subj:
        return "Subject"
    if obj:
        return "Object"
    return "Neither"


def vta_group(p):
    pred = p["predicate"]

    subj = pred["subject_agreement"] in AN_THIRD_PERSON
    obj = pred["object_agreement"] in AN_THIRD_PERSON

    if subj and obj:
        return "Non-local"
    if not subj and not obj:
        return "Local"
    return "Mixed"


def direction(p):
    pred = p["predicate"]

    subj = pred.get("subject_agreement") or ""
    obj = pred.get("object_agreement") or ""

    if not subj or not obj:
        return None

    if subj == obj:
        return "equal"

    s = PERSON_HIERARCHY.get(subj[0])
    o = PERSON_HIERARCHY.get(obj[0])

    if s is None or o is None:
        return None

    if s < o:
        return "direct"
    if s > o:
        return "inverse"

    if "Prox" in subj and "Obv" in obj:
        return "direct"
    if "Obv" in subj and "Prox" in obj:
        return "inverse"

    return None


def query(
    predicates,
    paradigm=None,
    group=None,
    pattern=None,
    inflection=None,
    voice=None,
):
    return [
        p for p in predicates
        if (paradigm is None or p["predicate"]["verb_paradigm"] == paradigm)
        and (group is None or vta_group(p) == group)
        and (pattern is None or argument_pattern(p) == pattern)
        and (inflection is None or p["predicate"]["order"] == inflection)
        and (voice is None or direction(p) == voice)
    ]


def word_order(p, pattern=None):
    order = p["head_order"]

    if "I" in order:
        return "Indirect object"

    if pattern is not None and order not in WORD_ORDERS[pattern]:
        return "Other"

    return order

def print_word_orders(predicates, pattern):
    counts = Counter(
        (word_order(p, pattern), direction(p))
        for p in predicates
    )

    orders = (
        *WORD_ORDERS[pattern],
        "Indirect object",
        "Other",
    )

    print(f"{'Order':<22}{'Direct':>8}{'Inverse':>10}{'Equal':>10}")

    for order in orders:
        d = counts[(order, "direct")]
        i = counts[(order, "inverse")]
        u = counts[(order, "equal")]

        if d + i + u:
            print(f"{order:<22}{d:>8}{i:>10}{u:>10}")

    assert sum(counts.values()) == len(predicates)

In [7]:
vtas = query(all_predicates, paradigm="VTA")

assert sum(len(query(vtas, group=g)) for g in VTA_GROUPS) == len(vtas)

for group in VTA_GROUPS:
    predicates = query(vtas, group=group)

    print(f"\n{group} VTAs (n={len(predicates)})")
    print(f"{'Pattern':<12}{'Total':>8}{'Direct':>10}{'Inverse':>10}{'Equal':>10}")

    for pattern in ARGUMENT_PATTERNS:
        subset = query(predicates, pattern=pattern)
        counts = Counter(direction(p) for p in subset)

        print(
            f"{pattern:<12}{len(subset):>8}"
            f"{counts['direct']:>10}{counts['inverse']:>10}{counts['equal']:>10}"
        )

        assert sum(counts.values()) == len(subset)

    assert sum(
        len(query(predicates, pattern=p))
        for p in ARGUMENT_PATTERNS
    ) == len(predicates)


Non-local VTAs (n=2433)
Pattern        Total    Direct   Inverse     Equal
Neither          913       763       139        11
Subject          274       161       107         6
Object          1016       991        25         0
Both             230       218        12         0

Local VTAs (n=787)
Pattern        Total    Direct   Inverse     Equal
Neither          732       159       571         2
Subject           19         3        16         0
Object            36         2        34         0
Both               0         0         0         0

Mixed VTAs (n=3454)
Pattern        Total    Direct   Inverse     Equal
Neither         1792      1318       474         0
Subject          317        26       291         0
Object          1296      1271        25         0
Both              49        37        12         0


In [8]:
for group in VTA_GROUPS:
    print(f"\n{'=' * 50}")
    print(f"{group} VTA word order")
    print(f"{'=' * 50}")

    for pattern in ("Subject", "Object", "Both"):
        predicates = query(vtas, group=group, pattern=pattern)

        print(f"\n{pattern} overt (n={len(predicates)})")
        print_word_orders(predicates, pattern)


Non-local VTA word order

Subject overt (n=274)
Order                   Direct   Inverse     Equal
SV                          89        31         1
VS                          71        73         5
Indirect object              1         3         0

Object overt (n=1016)
Order                   Direct   Inverse     Equal
VO                         812        13         0
OV                         170        12         0
Indirect object              9         0         0

Both overt (n=230)
Order                   Direct   Inverse     Equal
SVO                        115         2         0
SOV                          9         0         0
VSO                         19         4         0
VOS                         58         0         0
OSV                          4         2         0
OVS                          9         4         0
Indirect object              4         0         0

Local VTA word order

Subject overt (n=19)
Order                   Direct   Inverse     Equ

In [9]:
for paradigm in ("VTA", "VAI", "VII", "VTI"):
    groups = VTA_GROUPS if paradigm == "VTA" else (None,)

    for group in groups:
        for inflection in INFLECTIONAL_ORDERS:
            predicates = query(
                all_predicates,
                paradigm=paradigm,
                group=group,
                inflection=inflection,
            )

            if not predicates:
                continue

            label = f"{group} VTA" if group else paradigm
            print(f"\n{inflection} {label} (n={len(predicates)})")

            counts = Counter(argument_pattern(p) for p in predicates)

            if paradigm in {"VAI", "VII"}:
                overt = counts["Subject"] + counts["Both"]
                print(f"Subject overt: {overt}")
                print(f"Subject not overt: {len(predicates) - overt}")
            else:
                for pattern in ARGUMENT_PATTERNS:
                    n = counts[pattern]
                    print(f"{pattern}: {n} ({100*n/len(predicates):.1f}%)")

            assert sum(counts.values()) == len(predicates)


Ind Non-local VTA (n=1102)
Neither: 313 (28.4%)
Subject: 133 (12.1%)
Object: 509 (46.2%)
Both: 147 (13.3%)

Cnj Non-local VTA (n=1216)
Neither: 508 (41.8%)
Subject: 130 (10.7%)
Object: 495 (40.7%)
Both: 83 (6.8%)

Pcp Non-local VTA (n=115)
Neither: 92 (80.0%)
Subject: 11 (9.6%)
Object: 12 (10.4%)
Both: 0 (0.0%)

Ind Local VTA (n=399)
Neither: 364 (91.2%)
Subject: 14 (3.5%)
Object: 21 (5.3%)
Both: 0 (0.0%)

Cnj Local VTA (n=309)
Neither: 291 (94.2%)
Subject: 5 (1.6%)
Object: 13 (4.2%)
Both: 0 (0.0%)

Imp Local VTA (n=79)
Neither: 77 (97.5%)
Subject: 0 (0.0%)
Object: 2 (2.5%)
Both: 0 (0.0%)

Ind Mixed VTA (n=1402)
Neither: 624 (44.5%)
Subject: 152 (10.8%)
Object: 599 (42.7%)
Both: 27 (1.9%)

Cnj Mixed VTA (n=1586)
Neither: 931 (58.7%)
Subject: 160 (10.1%)
Object: 475 (29.9%)
Both: 20 (1.3%)

Imp Mixed VTA (n=339)
Neither: 128 (37.8%)
Subject: 0 (0.0%)
Object: 211 (62.2%)
Both: 0 (0.0%)

Pcp Mixed VTA (n=127)
Neither: 109 (85.8%)
Subject: 5 (3.9%)
Object: 11 (8.7%)
Both: 2 (1.6%)

Ind VA

In [10]:
for paradigm in ("VTA", "VAI", "VII", "VTI"):
    groups = VTA_GROUPS if paradigm == "VTA" else (None,)

    for group in groups:
        for inflection in INFLECTIONAL_ORDERS:
            predicates = query(
                all_predicates,
                paradigm=paradigm,
                group=group,
                inflection=inflection,
            )

            if not predicates:
                continue

            label = f"{group} VTA" if group else paradigm
            print(f"\n{'=' * 50}")
            print(f"{inflection} {label} (n={len(predicates)})")

            if paradigm == "VTA":
                for pattern in ("Subject", "Object", "Both"):
                    subset = query(predicates, pattern=pattern)

                    if subset:
                        print(f"\n{pattern} overt (n={len(subset)})")
                        print_word_orders(subset, pattern)

            else:
                counts = Counter(
                    "Indirect object" if "I" in p["head_order"]
                    else p["head_order"]
                    for p in predicates
                )

                for order, n in sorted(counts.items()):
                    print(
                        f"{order}: {n} "
                        f"({100*n/len(predicates):.1f}%)"
                    )

                assert sum(counts.values()) == len(predicates)


Ind Non-local VTA (n=1102)

Subject overt (n=133)
Order                   Direct   Inverse     Equal
SV                          34        20         0
VS                          21        52         2
Indirect object              1         3         0

Object overt (n=509)
Order                   Direct   Inverse     Equal
VO                         411         9         0
OV                          82         4         0
Indirect object              3         0         0

Both overt (n=147)
Order                   Direct   Inverse     Equal
SVO                         72         2         0
SOV                          9         0         0
VSO                         12         4         0
VOS                         38         0         0
OSV                          2         2         0
OVS                          3         2         0
Indirect object              1         0         0

Cnj Non-local VTA (n=1216)

Subject overt (n=130)
Order                   Direct   Inverse